# AlexNet-Style CNN on CIFAR-10This Colab-ready notebook implements a compact AlexNet-inspired convolutional neural network for CIFAR-10 image classification. The architecture and training strategy are informed by the ideas introduced in *ImageNet Classification with Deep Convolutional Neural Networks* (Krizhevsky et al., 2012).

## Environment SetupThis notebook is designed for Google Colab. A GPU runtime is recommended (`Runtime > Change runtime type > GPU`). The required libraries are part of the standard Colab image.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Data PipelineWe use CIFAR-10 as a manageable proxy for ImageNet. Normalization statistics are taken from the dataset, and the augmentation strategy (random crops, horizontal flips) mirrors the data augmentation described in the AlexNet paper.

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 2

mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_dataset = datasets.CIFAR10(root='data', train=True, download=True, transform=train_transforms)
val_dataset = datasets.CIFAR10(root='data', train=False, download=True, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

classes = train_dataset.classes
print(f"Classes: {classes}")

## AlexNet-Inspired ModelThe original AlexNet operates on 224×224 ImageNet images and uses grouped convolutions to distribute the network across two GPUs. For CIFAR-10 we:* Use smaller convolutional kernels and fewer channels while retaining the core pattern of convolution → ReLU → pooling.* Maintain local response normalization (LRN) layers conceptually by using batch normalization for better stability.* Introduce dropout in the fully connected layers to regularize the network as done in the paper.

In [ ]:
class AlexNetCIFAR(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(192),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model = AlexNetCIFAR(num_classes=len(classes)).to(device)
print(model)

## Training UtilitiesWe implement standard training and evaluation loops with learning-rate scheduling. The optimizer and hyperparameters mirror those used in the original paper (SGD with momentum, weight decay).

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = StepLR(optimizer, step_size=30, gamma=0.1)

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc='Train', leave=False)
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        pbar.set_postfix(loss=running_loss / total, acc=100.0 * correct / total)

    return running_loss / total, correct / total

def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    return running_loss / total, correct / total

## Training LoopTrain for 50 epochs (adjust as desired). The scheduler decays the learning rate every 30 epochs as in the original paper.

In [ ]:
EPOCHS = 50

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch}/{EPOCHS}")
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%")

## Plot MetricsVisualize the training and validation curves to monitor convergence.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss Curves')

plt.subplot(1,2,2)
plt.plot(np.array(history['train_acc']) * 100, label='Train Acc')
plt.plot(np.array(history['val_acc']) * 100, label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.title('Accuracy Curves')

plt.show()

## Inspect PredictionsSample a few predictions to qualitatively assess model performance.

In [ ]:
model.eval()
images, labels = next(iter(val_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    _, predicted = outputs.max(1)

images = images.cpu()
labels = labels.cpu()
predicted = predicted.cpu()

fig, axes = plt.subplots(2, 5, figsize=(15,6))
for idx, ax in enumerate(axes.flatten()):
    img = images[idx]
    img = img * torch.tensor(std).view(3,1,1) + torch.tensor(mean).view(3,1,1)
    img = torch.clamp(img, 0, 1)
    img = np.transpose(img.numpy(), (1, 2, 0))

    ax.imshow(img)
    ax.set_title(f"Pred: {classes[predicted[idx]]}\nTrue: {classes[labels[idx]]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

## Save the ModelExport the trained model's weights for reuse.

In [ ]:
MODEL_PATH = 'alexnet_cifar10.pth'
torch.save(model.state_dict(), MODEL_PATH)
print(f"Saved model weights to {MODEL_PATH}")

## Next Steps* Replace CIFAR-10 with a higher-resolution dataset (e.g., Tiny ImageNet) to further align with the original paper's scale.* Experiment with deeper architectures (e.g., VGG, ResNet) and compare accuracy.* Apply transfer learning using pretrained ImageNet weights.